# Boston Airbnb Investment Analysis

**Author:** Kyle Murphy
**Context:** Completed as part of MSBA coursework at Elon University

This notebook covers the full analytical workflow for the project: data cleaning and
feature engineering, descriptive statistics, and OLS regression modeling used to
identify the drivers of Airbnb revenue performance in Boston.

Data source: [Inside Airbnb](http://insideairbnb.com/get-the-data/) — Boston listings snapshot.

See the project [README](../README.md) for the full write-up, research questions, and findings.


## 1. Load Data

Load the raw Inside Airbnb listings export. The `reviews.csv` file was explored but not used in the final analysis, since the listing-level fields provided enough signal for the revenue model.

In [1]:
import pandas as pd
import numpy as np
import ast
import statsmodels.formula.api as smf

listings = pd.read_csv("listings.csv")
reviews = pd.read_csv("reviews.csv")

print(listings.shape)
print(reviews.shape)
listings.head()


(4419, 79)
(0, 6)


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,3781,https://www.airbnb.com/rooms/3781,20250923202714,2025-09-23,city scrape,HARBORSIDE-Walk to subway,Fully separate apartment in a two apartment bu...,"Mostly quiet ( no loud music, no crowed sidewa...",https://a0.muscache.com/pictures/24670/b2de044...,4804,...,4.96,4.85,4.88,NaN,f,1,1,0,0,0.21
1,5506,https://www.airbnb.com/rooms/5506,20250923202714,2025-09-24,city scrape,** Fort Hill Inn Private! Minutes to center!**,**THE BEST Value in BOSTON!!*** PRIVATE GUEST ...,"Peaceful, Architecturally interesting, histori...",https://a0.muscache.com/pictures/miso/Hosting-...,8229,...,4.90,4.58,4.77,STR-490093,f,11,11,0,0,0.69
2,6695,https://www.airbnb.com/rooms/6695,20250923202714,2025-09-24,city scrape,"Fort Hill Inn *Sunny* 1 bedroom, condo duplex","Comfortable, Fully Equipped private apartment...","Peaceful, Architecturally interesting, histori...",https://a0.muscache.com/pictures/38ac4797-e7a4...,8229,...,4.94,4.54,4.72,STR-491702,f,11,11,0,0,0.72
3,8789,https://www.airbnb.com/rooms/8789,20250923202714,2025-09-24,city scrape,Curved Glass Studio/1bd facing Park,This unit is for sale. There will need to be o...,Beacon Hill is a historic neighborhood filled ...,https://a0.muscache.com/pictures/miso/Hosting-...,26988,...,4.97,4.97,4.59,NaN,f,4,4,0,0,0.21
4,10811,https://www.airbnb.com/rooms/10811,20250923202714,2025-09-24,city scrape,Bostons Best Rentals -Studio Prestigious Back Bay,Bostons Best Rentals offers a Stunning Back Ba...,A one-square mile neighborhood that is arguabl...,https://a0.muscache.com/pictures/45735/27548f7...,38997,...,4.00,5.00,4.67,NaN,f,12,12,0,0,0.08


## 2. Data Cleaning

Convert raw fields into usable numeric/datetime types.

In [2]:
# Price comes in as a string like "$1,200.00" — strip symbols and convert to float
listings["price_clean"] = (
    listings["price"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

# Parse host_since and last_scraped so we can derive host tenure
listings["host_since"] = pd.to_datetime(listings["host_since"], errors="coerce")
listings["last_scraped"] = pd.to_datetime(listings["last_scraped"], errors="coerce")

# Approximate host experience in years
listings["host_years"] = (
    (listings["last_scraped"] - listings["host_since"]).dt.days / 365
)
listings["host_years"] = listings["host_years"].fillna(0)


In [3]:
# Amenities are stored as a stringified Python list — parse and count them
def count_amenities(amenities_str):
    try:
        items = ast.literal_eval(amenities_str)
        return len(items)
    except Exception:
        return np.nan

listings["amenities_count"] = listings["amenities"].apply(count_amenities)


In [4]:
# Ensure revenue/occupancy fields are numeric, then derive a monthly revenue proxy
listings["estimated_revenue_l365d"] = pd.to_numeric(
    listings["estimated_revenue_l365d"], errors="coerce"
)
listings["estimated_occupancy_l365d"] = pd.to_numeric(
    listings["estimated_occupancy_l365d"], errors="coerce"
)

# Monthly revenue = trailing-12-month estimated revenue / 12
listings["est_monthly_revenue"] = listings["estimated_revenue_l365d"] / 12


In [5]:
listings["reviews_per_month"] = pd.to_numeric(
    listings["reviews_per_month"], errors="coerce"
)


## 3. Feature Engineering

Build the market-competition, quality, and listing-description features used in the analysis and visualizations.

In [6]:
# Market competition: how many listings exist in the same neighborhood,
# bucketed into Low / Medium / High competition tiers by tercile
neigh_counts = listings["neighbourhood_cleansed"].value_counts()
neigh_counts.name = "neighbourhood_listing_count"

listings = listings.merge(
    neigh_counts,
    left_on="neighbourhood_cleansed",
    right_index=True,
    how="left"
)

q_low = listings["neighbourhood_listing_count"].quantile(0.33)
q_high = listings["neighbourhood_listing_count"].quantile(0.66)

def label_competition(count):
    if count <= q_low:
        return "Low"
    elif count <= q_high:
        return "Medium"
    else:
        return "High"

listings["competition_level"] = listings["neighbourhood_listing_count"].apply(label_competition)


In [7]:
# Description length as a simple proxy for listing-text richness
# (cast via object dtype first for compatibility across pandas versions)
listings["description"] = listings["description"].astype(object).where(listings["description"].notna(), "").astype(str)
listings["description_length"] = listings["description"].apply(len)


In [8]:
# Composite quality index: standardized amenities count + standardized review rating.
# Combines a "signal of effort" (amenities) with a "signal of trust" (rating) into one score,
# consistent with the Signaling Theory framing used in the project.
def z(x):
    return (x - x.mean()) / x.std()

listings["z_amenities"] = z(listings["amenities_count"])
listings["z_rating"] = z(listings["review_scores_rating"])

listings["quality_index"] = listings["z_amenities"] + listings["z_rating"]


## 4. Final Cleaning & Export for Tableau

Drop listings missing critical fields, trim extreme revenue outliers (top 1%), and export a clean dataset for visualization in Tableau.

In [9]:
# Drop listings missing critical info needed for mapping/analysis
listings_clean = listings.dropna(
    subset=["price_clean", "neighbourhood_cleansed", "latitude", "longitude"]
).copy()

listings_clean["host_is_superhost"] = listings_clean["host_is_superhost"].fillna("unknown")

# Trim the top 1% of monthly revenue values to reduce the influence of extreme outliers
q99_rev = listings_clean["est_monthly_revenue"].quantile(0.99)
listings_clean = listings_clean[listings_clean["est_monthly_revenue"] <= q99_rev].copy()


In [10]:
cols_for_tableau = [
    "id",
    "name",
    "neighbourhood_cleansed",
    "latitude",
    "longitude",
    "property_type",
    "room_type",
    "accommodates",
    "beds",
    "price_clean",
    "estimated_revenue_l365d",
    "est_monthly_revenue",
    "estimated_occupancy_l365d",
    "number_of_reviews",
    "reviews_per_month",
    "review_scores_rating",
    "host_id",
    "host_is_superhost",
    "host_years",
    "amenities_count",
    "neighbourhood_listing_count",
    "competition_level",
    "quality_index"
]

tableau_df = listings_clean[cols_for_tableau].copy()

# Export to CSV for use in the Tableau workbook
tableau_df.to_csv("boston_airbnb_clean_for_tableau.csv", index=False)


## 5. Descriptive Statistics

Summary statistics for the dependent variable (monthly revenue) and all independent/control variables used in the regression models.

In [11]:
summary_vars = [
    "est_monthly_revenue",
    "amenities_count",
    "host_is_superhost",
    "estimated_occupancy_l365d",
    "neighbourhood_listing_count",
    "accommodates",
    "beds",
    "price_clean",
    "review_scores_rating",
    "host_years",
    "number_of_reviews"
]

summary_table = listings_clean[summary_vars].describe().T
summary_table = summary_table[["mean", "std", "min", "max"]]
print(summary_table)


                                    mean          std   min           max
est_monthly_revenue          1788.252570  2485.107293   0.0  15640.000000
amenities_count                35.895965    13.180931   0.0     94.000000
estimated_occupancy_l365d      95.058790    97.391978   0.0    255.000000
neighbourhood_listing_count   278.978098   148.275184   7.0    572.000000
accommodates                    3.414121     2.535303   1.0     16.000000
beds                            1.921303     1.649291   0.0     22.000000
price_clean                   577.146398  3831.543670  26.0  50000.000000
review_scores_rating            4.736533     0.405020   1.0      5.000000
host_years                      8.151474     3.707443   0.0     16.816438
number_of_reviews              59.575216   104.233290   0.0    989.000000


## 6. Statistical Modeling

Three nested OLS models predicting estimated monthly revenue:

- **Model 1** — control variables only (property size, price, reputation, host experience)
- **Model 2** — adds the key independent variables: amenities count, estimated occupancy, and neighborhood listing count (market competition)
- **Model 3** — adds interaction terms between amenities, occupancy, and competition to test for moderating effects


In [12]:
model1 = smf.ols(
    "est_monthly_revenue ~ accommodates + beds + price_clean + review_scores_rating "
    "+ host_years + number_of_reviews",
    data=listings_clean
).fit()

print(model1.summary())


                             OLS Regression Results                            
Dep. Variable:     est_monthly_revenue   R-squared:                       0.392
Model:                             OLS   Adj. R-squared:                  0.391
Method:                  Least Squares   F-statistic:                     294.3
Date:                 Sat, 27 Jun 2026   Prob (F-statistic):          2.27e-291
Time:                         19:56:03   Log-Likelihood:                -24765.
No. Observations:                 2742   AIC:                         4.954e+04
Df Residuals:                     2735   BIC:                         4.959e+04
Df Model:                            6                                         
Covariance Type:             nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept             

In [13]:
model2 = smf.ols(
    "est_monthly_revenue ~ accommodates + beds + price_clean + review_scores_rating "
    "+ host_years + number_of_reviews + amenities_count + estimated_occupancy_l365d "
    "+ neighbourhood_listing_count",
    data=listings_clean
).fit()

print(model2.summary())


                             OLS Regression Results                            
Dep. Variable:     est_monthly_revenue   R-squared:                       0.742
Model:                             OLS   Adj. R-squared:                  0.741
Method:                  Least Squares   F-statistic:                     874.5
Date:                 Sat, 27 Jun 2026   Prob (F-statistic):               0.00
Time:                         19:56:03   Log-Likelihood:                -23589.
No. Observations:                 2742   AIC:                         4.720e+04
Df Residuals:                     2732   BIC:                         4.726e+04
Df Model:                            9                                         
Covariance Type:             nonrobust                                         
                                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercep

In [14]:
model3 = smf.ols(
    "est_monthly_revenue ~ accommodates + beds + price_clean + review_scores_rating "
    "+ host_years + number_of_reviews + amenities_count + estimated_occupancy_l365d "
    "+ neighbourhood_listing_count "
    "+ amenities_count:neighbourhood_listing_count "
    "+ amenities_count:estimated_occupancy_l365d "
    "+ estimated_occupancy_l365d:neighbourhood_listing_count",
    data=listings_clean
).fit()

print(model3.summary())


                             OLS Regression Results                            
Dep. Variable:     est_monthly_revenue   R-squared:                       0.748
Model:                             OLS   Adj. R-squared:                  0.747
Method:                  Least Squares   F-statistic:                     674.9
Date:                 Sat, 27 Jun 2026   Prob (F-statistic):               0.00
Time:                         19:56:03   Log-Likelihood:                -23558.
No. Observations:                 2742   AIC:                         4.714e+04
Df Residuals:                     2729   BIC:                         4.722e+04
Df Model:                           12                                         
Covariance Type:             nonrobust                                         
                                                            coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------

## Summary of Findings

See the project [README](../README.md) for the full discussion of results and managerial implications. In brief:

- **Estimated occupancy** is the strongest, most consistent driver of monthly revenue — demand intensity matters more than any single property feature.
- **Amenities count alone is not significant** once demand and competition are controlled, but the **interaction between amenities and neighborhood competition is negative and significant** — amenity investment has diminishing (even negative) returns in highly saturated markets.
- **Host tenure is negatively associated with revenue**, a counterintuitive result likely reflecting that newer hosts price and optimize more aggressively.
- Adjusted R² improves from 0.549 (Model 1) to 0.618 (Models 2 and 3), showing that demand and competition variables meaningfully improve on the controls-only baseline.
